In [1]:
# CELL 1: CÀI ĐẶT THƯ VIỆN
# (Trên Kaggle dùng pip trực tiếp trong notebook — Poetry chỉ dùng cho local dev)
!pip install -q -U accelerate transformers peft bitsandbytes trl datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 90.9 MB/s eta 0:00:00


In [2]:
# CELL 2: IMPORT VÀ CẤU HÌNH HẰNG SỐ
import os
import gc
import math
import logging
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model
import matplotlib
matplotlib.use("Agg")  # Non-interactive backend for Kaggle
import matplotlib.pyplot as plt

# ---------- Logging ----------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)

# ---------- Hằng số của project (mirrors config.py) ----------
MODEL_ID        = os.getenv("QWEN_MODEL_ID", "Qwen/Qwen2.5-Math-1.5B-Instruct")
DATA_PATH       = "/kaggle/input/datasets/hauuto/vietnamese-elementary-math/data_warehouse.csv"
OUTPUT_DIR      = "/kaggle/working/models/m4_qwen"
CHECKPOINT_DIR  = f"{OUTPUT_DIR}/checkpoints"
FINAL_MODEL_DIR = f"{OUTPUT_DIR}/final"
LOGS_DIR        = f"{OUTPUT_DIR}/logs"
FIGURES_DIR     = f"{OUTPUT_DIR}/figures"

# QLoRA hyperparams (mirrors config.py defaults)
QLORA_R        = int(os.getenv("QLORA_R", "16"))
QLORA_ALPHA    = int(os.getenv("QLORA_ALPHA", "32"))
QLORA_DROPOUT  = float(os.getenv("QLORA_DROPOUT", "0.05"))
TRAIN_BATCH_SZ = int(os.getenv("TRAIN_BATCH_SIZE", "1"))
GRAD_ACCUM     = int(os.getenv("GRAD_ACCUM_STEPS", "4"))
MAX_SEQ_LEN    = 256   # Giới hạn token để bảo vệ VRAM trên T4
MAX_TRAIN_SAMPLES = 20000  # Cắt tập train để vừa thời gian Kaggle

for d in [CHECKPOINT_DIR, FINAL_MODEL_DIR, LOGS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

logger.info(f"Model ID  : {MODEL_ID}")
logger.info(f"Data path : {DATA_PATH}")
logger.info(f"Output dir: {OUTPUT_DIR}")

2026-05-01 15:44:37,377 [INFO] Model ID  : Qwen/Qwen2.5-Math-1.5B-Instruct
2026-05-01 15:44:37,377 [INFO] Data path : /kaggle/input/datasets/hauuto/vietnamese-elementary-math/data_warehouse.csv
2026-05-01 15:44:37,378 [INFO] Output dir: /kaggle/working/models/m4_qwen


In [3]:
# CELL 3: ĐỌC VÀ TIỀN XỬ LÝ DỮ LIỆU
# Sử dụng Qwen2.5 Chat Template thay vì Gemma turn format

logger.info(f"Đang đọc dữ liệu từ: {DATA_PATH}")
try:
    df = pd.read_csv(DATA_PATH)
    df = df.dropna(subset=["question", "answer"])
    logger.info(f"Tổng số dòng hợp lệ trước khi chia: {len(df)}")
except FileNotFoundError:
    logger.warning(f"Không tìm thấy {DATA_PATH}. Tạo dữ liệu giả để test pipeline.")
    df = pd.DataFrame({
        "instruction": ["Hãy giải bài toán sau từng bước:"],
        "question":    ["Một túi có 5 viên bi. Thêm 3 viên nữa. Túi có bao nhiêu viên?"],
        "answer":      ["Bước 1: 5 + 3 = 8.\nVậy túi có 8 viên bi."],
    })


def format_qwen_chat(row: pd.Series) -> str:
    """
    Format a data row using Qwen2.5-Instruct chat template.
    Structure:
        <|im_start|>system\n{system}<|im_end|>
        <|im_start|>user\n{instruction}\n{question}<|im_end|>
        <|im_start|>assistant\n{answer}<|im_end|>
    """
    instruction = row.get("instruction")
    # Làm sạch instruction bị null / rác
    if pd.isna(instruction) or str(instruction).strip() in ["", "[null]", "[]", "main/train"]:
        instruction = "Hãy giải bài toán sau từng bước, trình bày rõ ràng từng bước tính:"

    system_msg = (
        "Bạn là một trợ lý chuyên giải toán tiểu học tiếng Việt. "
        "Hãy trình bày lời giải từng bước một cách rõ ràng, dễ hiểu cho học sinh."
    )
    question = str(row["question"]).strip()
    answer   = str(row["answer"]).strip()

    return (
        f"<|im_start|>system\n{system_msg}<|im_end|>\n"
        f"<|im_start|>user\n{instruction}\n{question}<|im_end|>\n"
        f"<|im_start|>assistant\n{answer}<|im_end|>"
    )


df["text"] = df.apply(format_qwen_chat, axis=1)

# Chuyển sang HuggingFace Dataset
full_dataset = Dataset.from_pandas(df[["text"]])

# Chia tập dữ liệu: 90% train / 5% val / 5% test  (seed=42 để tái tạo được)
split_90_10  = full_dataset.train_test_split(test_size=0.10, seed=42)
train_dataset = split_90_10["train"]
temp_dataset  = split_90_10["test"]

split_5_5    = temp_dataset.train_test_split(test_size=0.50, seed=42)
eval_dataset = split_5_5["train"]
test_dataset = split_5_5["test"]

# Cắt tập train xuống MAX_TRAIN_SAMPLES để fit trong giới hạn thời gian Kaggle
if len(train_dataset) > MAX_TRAIN_SAMPLES:
    train_dataset = train_dataset.select(range(MAX_TRAIN_SAMPLES))

logger.info("-" * 40)
logger.info(f"Train  (đã rút gọn) : {len(train_dataset):,}")
logger.info(f"Validation           : {len(eval_dataset):,}")
logger.info(f"Test                 : {len(test_dataset):,}")
logger.info("-" * 40)

# Lưu tập test để dùng sau
test_dataset.to_pandas().to_csv(f"{OUTPUT_DIR}/test_set.csv", index=False)
logger.info(f"Đã lưu test set tại: {OUTPUT_DIR}/test_set.csv")

2026-05-01 15:44:37,399 [INFO] Đang đọc dữ liệu từ: /kaggle/input/datasets/hauuto/vietnamese-elementary-math/data_warehouse.csv
2026-05-01 15:44:39,633 [INFO] Tổng số dòng hợp lệ trước khi chia: 96502
2026-05-01 15:44:41,708 [INFO] ----------------------------------------
2026-05-01 15:44:41,709 [INFO] Train  (đã rút gọn) : 20,000
2026-05-01 15:44:41,709 [INFO] Validation           : 4,825
2026-05-01 15:44:41,710 [INFO] Test                 : 4,826
2026-05-01 15:44:41,711 [INFO] ----------------------------------------
2026-05-01 15:44:41,897 [INFO] Đã lưu test set tại: /kaggle/working/models/m4_qwen/test_set.csv


In [4]:
# CELL 4: NẠP MÔ HÌNH VÀ CẤU HÌNH QLoRA
logger.info("--- BƯỚC 2: NẠP TOKENIZER VÀ MÔ HÌNH NỀN ---")

torch.cuda.empty_cache()
gc.collect()

# 4-bit quantization config (NF4 + double quant)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# Tokenizer — Qwen2.5 sử dụng padding bên phải
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "right"   # Quan trọng với Qwen2.5
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

logger.info(f"Tokenizer loaded. vocab_size={tokenizer.vocab_size:,}")

# Nạp base model (device_map=auto sẽ phân bổ qua 2 GPU T4 nếu có)
logger.info("Đang nạp trọng số mô hình... (có thể mất 1–2 phút)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,   # Cần thiết cho Qwen
)
model.config.use_cache = False   # Tắt cache khi train

logger.info("Đang áp dụng LoRA adapters...")
lora_config = LoraConfig(
    r=QLORA_R,
    lora_alpha=QLORA_ALPHA,
    lora_dropout=QLORA_DROPOUT,
    target_modules="all-linear",  # Áp dụng LoRA lên tất cả linear layers
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
logger.info("Số lượng tham số Trainable:")
model.print_trainable_parameters()

2026-05-01 15:44:41,914 [INFO] --- BƯỚC 2: NẠP TOKENIZER VÀ MÔ HÌNH NỀN ---
2026-05-01 15:44:42,323 [INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Math-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-01 15:44:42,342 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Math-1.5B-Instruct/aafeb0fc6f22cbf0eaeed126eff8be45b0360a35/config.json "HTTP/1.1 200 OK"
2026-05-01 15:44:42,362 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Math-1.5B-Instruct/aafeb0fc6f22cbf0eaeed126eff8be45b0360a35/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/656 [00:00<?, ?B/s]

2026-05-01 15:44:42,445 [INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Math-1.5B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-01 15:44:42,463 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Math-1.5B-Instruct/aafeb0fc6f22cbf0eaeed126eff8be45b0360a35/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-01 15:44:42,482 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Math-1.5B-Instruct/aafeb0fc6f22cbf0eaeed126eff8be45b0360a35/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json: 0.00B [00:00, ?B/s]

2026-05-01 15:44:42,561 [INFO] HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-Math-1.5B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-01 15:44:42,629 [INFO] HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-Math-1.5B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-05-01 15:44:42,693 [INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Math-1.5B-Instruct/resolve/main/vocab.json "HTTP/1.1 307 Temporary Redirect"
2026-05-01 15:44:42,695 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-01 15:44:42,713 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Math-1.5B-Instruct/aafeb0fc6f22cbf0eaeed126eff8be45b0360a35/vocab.json "HTTP/1.1 200 OK"
2026-05-01 15:44:42,733 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache

...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

2026-05-01 15:45:04,718 [INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Math-1.5B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-01 15:45:04,735 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Math-1.5B-Instruct/aafeb0fc6f22cbf0eaeed126eff8be45b0360a35/generation_config.json "HTTP/1.1 200 OK"
2026-05-01 15:45:04,754 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Math-1.5B-Instruct/aafeb0fc6f22cbf0eaeed126eff8be45b0360a35/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

2026-05-01 15:45:04,829 [INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Math-1.5B-Instruct/resolve/main/custom_generate/generate.py "HTTP/1.1 404 Not Found"
2026-05-01 15:45:05,035 [INFO] Đang áp dụng LoRA adapters...
2026-05-01 15:45:05,515 [INFO] Số lượng tham số Trainable:


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [5]:
# CELL 5: TOKENIZE DỮ LIỆU
logger.info("--- BƯỚC 3: TOKENIZE DỮ LIỆU (max_length={}) ---".format(MAX_SEQ_LEN))


def tokenize_function(examples):
    """Tokenize and truncate to MAX_SEQ_LEN to protect VRAM."""
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,   # DataCollator sẽ xử lý padding động
    )


cols_to_remove = train_dataset.column_names

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=cols_to_remove,
    desc="Tokenizing train",
)
tokenized_eval = eval_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=cols_to_remove,
    desc="Tokenizing eval",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

logger.info(f"Train tokenized : {len(tokenized_train):,} samples")
logger.info(f"Eval tokenized  : {len(tokenized_eval):,} samples")

2026-05-01 15:45:05,543 [INFO] --- BƯỚC 3: TOKENIZE DỮ LIỆU (max_length=256) ---


Tokenizing train:   0%|          | 0/20000 [00:00<?, ? examples/s]

Tokenizing eval:   0%|          | 0/4825 [00:00<?, ? examples/s]

2026-05-01 15:45:17,648 [INFO] Train tokenized : 20,000 samples
2026-05-01 15:45:17,649 [INFO] Eval tokenized  : 4,825 samples


In [6]:
# CELL 6: HUẤN LUYỆN VỚI CHECKPOINTING
logger.info("--- BƯỚC 4: KHỞI ĐỘNG HUẤN LUYỆN ---")

model.enable_input_require_grads()
model.gradient_checkpointing_enable()

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,

    # Epochs & batch
    num_train_epochs=1,
    per_device_train_batch_size=TRAIN_BATCH_SZ,      # =1 để bảo vệ VRAM T4
    gradient_accumulation_steps=GRAD_ACCUM,          # Effective batch = 1×4 = 4

    # Learning rate schedule
    learning_rate=2e-4,          # Qwen2.5-Math phản hồi tốt với LR cao hơn Gemma
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,

    # Precision & optimizer
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",
    weight_decay=0.01,

    # Gradient checkpointing (tiết kiệm VRAM đáng kể)
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # Logging
    logging_steps=10,
    report_to="none",

    # Evaluation (tắt trong lúc train để tiết kiệm thời gian Kaggle)
    eval_strategy="no",

    # Checkpoint — lưu mỗi 200 steps, giữ 2 bản gần nhất
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

torch.cuda.empty_cache()
gc.collect()

logger.info("Bắt đầu quá trình train...")
train_result = trainer.train()

logger.info(f"Training hoàn tất. Runtime: {train_result.metrics.get('train_runtime', 0):.0f}s")
logger.info(f"Samples/s: {train_result.metrics.get('train_samples_per_second', 0):.2f}")

2026-05-01 15:45:17,670 [INFO] --- BƯỚC 4: KHỞI ĐỘNG HUẤN LUYỆN ---
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
2026-05-01 15:45:18,042 [INFO] Bắt đầu quá trình train...


Step,Training Loss
10,6.923030
20,6.606254
30,6.227317
40,5.742859
50,5.233712
60,4.213581
70,3.590909
80,3.122009
90,3.042932
100,2.508888


2026-05-01 16:03:58,059 [INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Math-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-01 16:03:58,076 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Math-1.5B-Instruct/aafeb0fc6f22cbf0eaeed126eff8be45b0360a35/config.json "HTTP/1.1 200 OK"
2026-05-01 16:03:58,137 [INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Math-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-01 16:03:58,154 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Math-1.5B-Instruct/aafeb0fc6f22cbf0eaeed126eff8be45b0360a35/config.json "HTTP/1.1 200 OK"
2026-05-01 16:22:44,339 [INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Math-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-01 16:22:44,355 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwe

In [7]:
# CELL 7: LƯU MÔ HÌNH VÀ TOKENIZER
logger.info("--- BƯỚC 5: LƯU MÔ HÌNH HOÀN CHỈNH ---")

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

logger.info(f"Model và Tokenizer đã lưu tại: {FINAL_MODEL_DIR}")
logger.info("Nội dung thư mục final:")
for f in os.listdir(FINAL_MODEL_DIR):
    logger.info(f"  {f}")

2026-05-01 23:32:42,823 [INFO] --- BƯỚC 5: LƯU MÔ HÌNH HOÀN CHỈNH ---
2026-05-01 23:32:42,936 [INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Math-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-01 23:32:42,954 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Math-1.5B-Instruct/aafeb0fc6f22cbf0eaeed126eff8be45b0360a35/config.json "HTTP/1.1 200 OK"
2026-05-01 23:32:43,014 [INFO] HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Math-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-01 23:32:43,031 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Math-1.5B-Instruct/aafeb0fc6f22cbf0eaeed126eff8be45b0360a35/config.json "HTTP/1.1 200 OK"
2026-05-01 23:32:43,374 [INFO] Model và Tokenizer đã lưu tại: /kaggle/working/models/m4_qwen/final
2026-05-01 23:32:43,375 [INFO] Nội dung thư mục final:
2026-05-01 23:32:43,377 [INFO]   README.md
202

In [8]:
# CELL 8: XUẤT LOG HUẤN LUYỆN VÀ VẼ BIỂU ĐỒ LOSS
logger.info("--- BƯỚC 6: XUẤT LOG VÀ BIỂU ĐỒ ---")

# Trích log history từ trainer state
log_history = trainer.state.log_history
log_df = pd.DataFrame(log_history)

# Tách train loss và eval loss (nếu có)
train_logs = log_df.dropna(subset=["loss"]).copy() if "loss" in log_df.columns else pd.DataFrame()
eval_logs  = log_df.dropna(subset=["eval_loss"]).copy() if "eval_loss" in log_df.columns else pd.DataFrame()

log_df.to_csv(f"{LOGS_DIR}/training_history.csv", index=False)
logger.info(f"Đã lưu log tại: {LOGS_DIR}/training_history.csv")

# Vẽ biểu đồ Train Loss
if not train_logs.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(train_logs["step"], train_logs["loss"], label="Train Loss", color="royalblue", linewidth=1.5)
    if not eval_logs.empty:
        ax.plot(eval_logs["step"], eval_logs["eval_loss"], label="Eval Loss",
                color="tomato", linewidth=1.5, linestyle="--", marker="o")
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.set_title("M4 — Qwen2.5-Math-1.5B QLoRA: Training Loss Curve")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig_path = f"{FIGURES_DIR}/loss_curve.png"
    fig.savefig(fig_path, dpi=150)
    plt.close(fig)
    logger.info(f"Biểu đồ loss đã lưu tại: {fig_path}")
else:
    logger.warning("Không có dữ liệu loss để vẽ biểu đồ.")

2026-05-01 23:32:43,415 [INFO] --- BƯỚC 6: XUẤT LOG VÀ BIỂU ĐỒ ---
2026-05-01 23:32:43,442 [INFO] Đã lưu log tại: /kaggle/working/models/m4_qwen/logs/training_history.csv
2026-05-01 23:32:43,714 [INFO] Biểu đồ loss đã lưu tại: /kaggle/working/models/m4_qwen/figures/loss_curve.png


In [9]:
# CELL 9: ĐÁNH GIÁ TRÊN TẬP TEST (LOSS & PERPLEXITY)
logger.info("--- BƯỚC 7: ĐÁNH GIÁ TRÊN TẬP TEST ---")

# Tokenize test set
tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=test_dataset.column_names,
    desc="Tokenizing test",
)

# Chuyển model sang chế độ inference
model.eval()
model.config.use_cache = True
torch.cuda.empty_cache()

test_results = trainer.evaluate(eval_dataset=tokenized_test)

try:
    test_results["perplexity"] = math.exp(test_results["eval_loss"])
except OverflowError:
    test_results["perplexity"] = float("inf")

logger.info(f"Test Loss        : {test_results['eval_loss']:.4f}")
logger.info(f"Perplexity       : {test_results['perplexity']:.4f}")

# Lưu metrics
pd.DataFrame([test_results]).to_csv(f"{LOGS_DIR}/test_metrics.csv", index=False)
logger.info(f"Test metrics lưu tại: {LOGS_DIR}/test_metrics.csv")

2026-05-01 23:32:43,751 [INFO] --- BƯỚC 7: ĐÁNH GIÁ TRÊN TẬP TEST ---


Tokenizing test:   0%|          | 0/4826 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step
0.732827,0.852717,5000


2026-05-02 00:06:27,548 [INFO] Test Loss        : 0.8527
2026-05-02 00:06:27,549 [INFO] Perplexity       : 2.3460
2026-05-02 00:06:27,551 [INFO] Test metrics lưu tại: /kaggle/working/models/m4_qwen/logs/test_metrics.csv


In [10]:
# CELL 10: INFERENCE THỬ NGHIỆM TRÊN 10 MẪU NGẪU NHIÊN
logger.info("--- BƯỚC 8: INFERENCE SAMPLE (10 mẫu ngẫu nhiên) ---")

sample_test = test_dataset.shuffle(seed=42).select(range(min(10, len(test_dataset))))
inference_results = []

with torch.no_grad():
    for i, item in enumerate(sample_test):
        # Tách prompt (user turn) và ground truth (assistant turn)
        raw_text: str = item["text"]
        split_marker = "<|im_start|>assistant\n"
        if split_marker in raw_text:
            prompt       = raw_text.split(split_marker)[0] + split_marker
            ground_truth = raw_text.split(split_marker)[1].replace("<|im_end|>", "").strip()
        else:
            prompt       = raw_text
            ground_truth = ""

        # Tokenize prompt
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        # Generate — giới hạn 150 new tokens để tránh OOM
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1,       # Thấp hơn M3: Qwen2.5-Math ưa deterministic
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        # Decode (bỏ phần prompt, chỉ lấy phần mô hình sinh ra)
        generated_text = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        ).strip()

        del inputs, outputs
        torch.cuda.empty_cache()

        inference_results.append({
            "Prompt":       prompt,
            "Ground_Truth": ground_truth,
            "AI_Prediction": generated_text,
        })
        logger.info(f"Đã sinh xong mẫu {i+1}/10")

# Lưu kết quả inference
inference_df = pd.DataFrame(inference_results)
inference_csv = f"{OUTPUT_DIR}/inference_sample_results.csv"
inference_df.to_csv(inference_csv, index=False)
logger.info(f"Kết quả inference lưu tại: {inference_csv}")

2026-05-02 00:06:27,585 [INFO] --- BƯỚC 8: INFERENCE SAMPLE (10 mẫu ngẫu nhiên) ---
2026-05-02 00:06:41,152 [INFO] Đã sinh xong mẫu 1/10
2026-05-02 00:07:01,717 [INFO] Đã sinh xong mẫu 2/10
2026-05-02 00:07:22,468 [INFO] Đã sinh xong mẫu 3/10
2026-05-02 00:07:39,515 [INFO] Đã sinh xong mẫu 4/10
2026-05-02 00:08:00,321 [INFO] Đã sinh xong mẫu 5/10
2026-05-02 00:08:16,190 [INFO] Đã sinh xong mẫu 6/10
2026-05-02 00:08:36,973 [INFO] Đã sinh xong mẫu 7/10
2026-05-02 00:08:54,804 [INFO] Đã sinh xong mẫu 8/10
2026-05-02 00:08:59,194 [INFO] Đã sinh xong mẫu 9/10
2026-05-02 00:09:15,302 [INFO] Đã sinh xong mẫu 10/10
2026-05-02 00:09:15,305 [INFO] Kết quả inference lưu tại: /kaggle/working/models/m4_qwen/inference_sample_results.csv


In [11]:
# CELL 11: ĐÓNG GÓI KẾT QUẢ THÀNH FILE ZIP
import zipfile
import glob

logger.info("--- BƯỚC 9: ĐÓNG GÓI KẾT QUẢ ---")

zip_path = "/kaggle/working/m4_qwen_results.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    # Logs
    for csv_file in glob.glob(f"{LOGS_DIR}/*.csv"):
        zf.write(csv_file, os.path.relpath(csv_file, "/kaggle/working"))
    # Figures
    for fig_file in glob.glob(f"{FIGURES_DIR}/*.png"):
        zf.write(fig_file, os.path.relpath(fig_file, "/kaggle/working"))
    # Inference results
    if os.path.exists(inference_csv):
        zf.write(inference_csv, os.path.relpath(inference_csv, "/kaggle/working"))
    # Test set
    test_csv = f"{OUTPUT_DIR}/test_set.csv"
    if os.path.exists(test_csv):
        zf.write(test_csv, os.path.relpath(test_csv, "/kaggle/working"))

logger.info(f"✅ Đã đóng gói kết quả tại: {zip_path}")
logger.info("")
logger.info("📁 Tóm tắt output:")
logger.info(f"  Model final    : {FINAL_MODEL_DIR}")
logger.info(f"  Checkpoints    : {CHECKPOINT_DIR}")
logger.info(f"  Train log CSV  : {LOGS_DIR}/training_history.csv")
logger.info(f"  Test metrics   : {LOGS_DIR}/test_metrics.csv")
logger.info(f"  Loss curve     : {FIGURES_DIR}/loss_curve.png")
logger.info(f"  Inference CSV  : {inference_csv}")
logger.info(f"  Result ZIP     : {zip_path}")
logger.info("")
logger.info("🎉 M4 training pipeline hoàn tất!")

2026-05-02 00:09:15,340 [INFO] --- BƯỚC 9: ĐÓNG GÓI KẾT QUẢ ---
2026-05-02 00:09:15,542 [INFO] ✅ Đã đóng gói kết quả tại: /kaggle/working/m4_qwen_results.zip
2026-05-02 00:09:15,543 [INFO] 
2026-05-02 00:09:15,543 [INFO] 📁 Tóm tắt output:
2026-05-02 00:09:15,544 [INFO]   Model final    : /kaggle/working/models/m4_qwen/final
2026-05-02 00:09:15,545 [INFO]   Checkpoints    : /kaggle/working/models/m4_qwen/checkpoints
2026-05-02 00:09:15,545 [INFO]   Train log CSV  : /kaggle/working/models/m4_qwen/logs/training_history.csv
2026-05-02 00:09:15,546 [INFO]   Test metrics   : /kaggle/working/models/m4_qwen/logs/test_metrics.csv
2026-05-02 00:09:15,547 [INFO]   Loss curve     : /kaggle/working/models/m4_qwen/figures/loss_curve.png
2026-05-02 00:09:15,547 [INFO]   Inference CSV  : /kaggle/working/models/m4_qwen/inference_sample_results.csv
2026-05-02 00:09:15,548 [INFO]   Result ZIP     : /kaggle/working/m4_qwen_results.zip
2026-05-02 00:09:15,549 [INFO] 
2026-05-02 00:09:15,549 [INFO] 🎉 M4 tra